# Homework 3 - Functions I

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
flights = pd.read_csv('../Data/nycflights.csv')
print(flights.shape)
flights.head()

(336776, 20)


,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,1,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00
1,2,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00
2,3,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00
3,4,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00
4,5,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00


## Q1. Write a function that returns the minimum, median, and maximum of a numerical variable

In [4]:
def summary_stats(x):
    """Returns the minimum, median, and maximum of a numerical variable."""
    minimum = x.min()
    median = x.median()
    maximum = x.max()
    return minimum, median, maximum

# Test with the month column
result = summary_stats(flights['month'])
print(f"Min: {result[0]}, Median: {result[1]}, Max: {result[2]}")

Min: 1, Median: 7.0, Max: 12


## Q2. Explain your reasoning for choosing your function's name

I named the function `summary_stats` because it computes summary statistics (minimum, median, and maximum) for a given numerical variable. The name is descriptive and clearly conveys what the function does.

## Q3. Write a function that categorizes a numerical variable into four time-of-day categories

In [5]:
def categorize_time(data, col_name):
    """
    Categorizes a military time column into four time-of-day categories:
    Morning (500-1159), Afternoon (1200-1659), Evening (1700-2059), Night (2100-459).
    """
    col = data[col_name]
    conditions = [
        (col >= 500) & (col <= 1159),
        (col >= 1200) & (col <= 1659),
        (col >= 1700) & (col <= 2059),
        (col >= 2100) | (col <= 459)
    ]
    choices = ['Morning', 'Afternoon', 'Evening', 'Night']
    return np.select(conditions, choices, default=np.nan)

# Test with dep_time
flights['dep_time_cat'] = categorize_time(flights, 'dep_time')
print(flights[['dep_time', 'dep_time_cat']].head())
print(flights[['dep_time', 'dep_time_cat']].tail())
print("\nFrequency table:")
print(flights['dep_time_cat'].value_counts())

   dep_time dep_time_cat
0     517.0      Morning
1     533.0      Morning
2     542.0      Morning
3     544.0      Morning
4     554.0      Morning
        dep_time dep_time_cat
336771       NaN          nan
336772       NaN          nan
336773       NaN          nan
336774       NaN          nan
336775       NaN          nan

Frequency table:
Morning      129539
Afternoon     98617
Evening       79793
Night         20572
nan            8255
Name: dep_time_cat, dtype: int64


## Q4. Explain your reasoning for choosing your function's name

I named the function `categorize_time` because it categorizes a time variable (in military time format) into descriptive time-of-day categories. The name clearly indicates both the action (categorize) and the subject (time).

## Q5. Write a function that calculates the median of all numeric variables

In [6]:
def median_all_numeric(data):
    """Calculates the median of all numeric variables in a DataFrame."""
    numeric_data = data.select_dtypes(include='number')
    return numeric_data.median()

# Test with flights
result = median_all_numeric(flights)
print(result)

Unnamed: 0        168388.5
year                2013.0
month                  7.0
day                   16.0
dep_time            1401.0
sched_dep_time      1359.0
dep_delay             -2.0
arr_time            1535.0
sched_arr_time      1556.0
arr_delay             -5.0
flight              1496.0
air_time             129.0
distance             872.0
hour                  13.0
minute                29.0
dtype: float64


## Q6. Explain your reasoning for choosing your function's name

I named the function `median_all_numeric` because it calculates the median for all numeric columns in a given DataFrame. The name describes exactly what it does: computes the median across all numeric variables.

## Q7. Modify the t_test() function to handle HOV assumption violations

In [7]:
def t_test(data, num_vars, bin_vars):
    """
    Computes a t-test for the difference in means between two groups.
    Uses an independent samples t-test if HOV assumption holds,
    otherwise uses Welch's t-test.
    """
    # Check that all bin_vars are binary
    for bin_var in bin_vars:
        assert data[bin_var].nunique() == 2, f"'{bin_var}' must be dichotomous!"

    data = data[num_vars + bin_vars]
    results = []

    for num_var in num_vars:
        for bin_var in bin_vars:
            unique_vals = np.sort(data[bin_var].unique())

            group1 = data[num_var][data[bin_var] == unique_vals[0]].dropna()
            group2 = data[num_var][data[bin_var] == unique_vals[1]].dropna()

            # Sample statistics
            mean1, mean2 = group1.mean(), group2.mean()
            var1, var2 = group1.var(), group2.var()
            n1, n2 = len(group1), len(group2)
            mean_diff = mean2 - mean1

            # Check HOV assumption: variance ratio between 0.25 and 4.0
            var_ratio = var1 / var2 if var2 != 0 else np.inf

            if 0.25 <= var_ratio <= 4.0:
                # Independent samples t-test (HOV not violated)
                print("The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted")

                df = n1 + n2 - 2
                sp = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / df)
                se = sp * np.sqrt(1/n1 + 1/n2)
                t_stat = mean_diff / se
                p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df))
                test_name = "Independent samples t-test"
            else:
                # Welch's t-test (HOV violated)
                print("The homogeneity of variance assumption was violated, so Welch's t-test was conducted")

                se = np.sqrt(var1/n1 + var2/n2)
                t_stat = mean_diff / se
                df = (var1/n1 + var2/n2)**2 / (
                    (var1/n1)**2 / (n1 - 1) + (var2/n2)**2 / (n2 - 1)
                )
                p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df))
                test_name = "Welch's t-test"

            results.append({
                "Continuous Variable": num_var,
                "Binary Variable": bin_var,
                "Total Sample Size": n1 + n2,
                "Mean Difference": round(mean_diff, 2),
                "SE of Mean Difference": round(se, 2),
                "DF": round(df, 2),
                "t-statistic": round(t_stat, 3),
                "P-value": format(p_value, '.3f'),
                "test": test_name
            })

    return pd.DataFrame(results)

## Q8. Test the t_test() function on the GED dataset

In [9]:
# NOTE: Place the GED data file from the class activity in the Data folder
ged_df = pd.read_csv('../Data/3 ged_data.csv')
print(ged_df.head())
print(ged_df.tail())

   Unnamed: 0  STU_ID  SCH_ID  F3ERN2011  F3C02  F3EVRGED  F3EVERDO  BYMOTHED  \
0           2  101104    1011      37000     50         0         0         2   
1           5  101107    1011      35000     40         0         0         2   
2           7  101109    1011      68000     40         0         0         2   
3          10  101112    1011      18000      1         0         0         6   
4          18  101120    1011       1000     40         1         1         2   

   BYS14  BYRACE  ...  high_school_grad  ged  BYRACE2  Other  Asian  Black  \
0      2       7  ...                 1    0        7      0      0      0   
1      1       4  ...                 1    0        4      0      0      0   
2      1       7  ...                 1    0        7      0      0      0   
3      1       3  ...                 1    0        3      0      0      1   
4      1       7  ...                 0    1        7      0      0      0   

   Hispanic  White  post_sec_edu  income_log

In [10]:
results = t_test(data=ged_df, num_vars=['income_log'], bin_vars=['ged'])
print(results)

The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted
  Continuous Variable Binary Variable  Total Sample Size  Mean Difference  \
0          income_log             ged               5976            -0.48   

   SE of Mean Difference    DF  t-statistic P-value  \
0                   0.06  5974       -7.458   0.000   

                         test  
0  Independent samples t-test  
